# Objective: solve a genuine three-bus DCOPF fixture

Use the project's HiGHS-backed DCOPF API to solve a small network with optimizer-backed generation, bus angles, branch flows, thermal limits, and nodal balance equations.


In [ ]:
from pathlib import Path
import json, os, sys
from tools.activsg2000.dcopf import Branch, Bus, Generator, solve_dcopf

TUTORIAL_ROOT = Path(os.environ.get('TUTORIAL_ROOT', Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('TUTORIAL_OUTPUT_ROOT', TUTORIAL_ROOT / 'outputs')).resolve(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
buses = (Bus('1', 30.0, reference=True), Bus('2', 20.0), Bus('3', 10.0))
generators = (Generator('g1', '1', 0.0, 50.0, 10.0), Generator('g2', '3', 0.0, 40.0, 25.0))
branches = (Branch('1-2', '1', '2', 0.5, 25.0), Branch('2-3', '2', '3', 0.5, 25.0))


In [ ]:
print('Environment report')
print({'python': sys.version.split()[0], 'solver': 'tools.activsg2000.dcopf.solve_dcopf via HiGHS', 'fixture_first': True})


In [ ]:
result = solve_dcopf(buses=buses, generators=generators, branches=branches)
assert result.status == 'optimal'
assert result.angle_by_bus['1'] == 0.0
assert all(generator.p_min_mw <= result.generation_mw[generator.generator_id] <= generator.p_max_mw for generator in generators)
assert abs(sum(result.generation_mw.values()) - sum(bus.demand_mw for bus in buses)) < 1e-9
for branch in branches:
    expected_flow = branch.susceptance_pu * (result.angle_by_bus[branch.from_bus_id] - result.angle_by_bus[branch.to_bus_id])
    assert abs(result.flow_mw_by_branch[branch.branch_id] - expected_flow) < 1e-8
    assert abs(result.flow_mw_by_branch[branch.branch_id]) <= branch.thermal_limit_mw + 1e-9
balance = {bus.bus_id: result.generation_mw.get(next((generator.generator_id for generator in generators if generator.bus_id == bus.bus_id), ""), 0.0) - bus.demand_mw for bus in buses}
for branch in branches:
    balance[branch.from_bus_id] -= result.flow_mw_by_branch[branch.branch_id]
    balance[branch.to_bus_id] += result.flow_mw_by_branch[branch.branch_id]
assert all(abs(value) < 1e-8 for value in balance.values())
print('status:', result.status); print('objective_usd_per_hour:', result.objective_cost); print('generation_mw:', result.generation_mw); print('angle_by_bus:', result.angle_by_bus); print('branch_flows_mw:', result.flow_mw_by_branch)


In [ ]:
OUTPUT = OUTPUT_ROOT / 'three_bus_dcopf.json'
payload = {'status': result.status, 'objective_usd_per_hour': result.objective_cost, 'generation_mw': result.generation_mw, 'angle_by_bus': result.angle_by_bus, 'branch_flows_mw': result.flow_mw_by_branch, 'balance_mw': balance, 'note': 'Optimizer-backed educational three-bus DCOPF solved with the project HiGHS API.'}
OUTPUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('OUTPUT:', OUTPUT)
